# Weather Features — Open-Meteo (Lag 52 semanas)

Gera `features/ds_weather.parquet` com variáveis climáticas históricas por macro-região.

## Design: por que o lag de 52 semanas?

| Cenário | Problema |
|---|---|
| Clima futuro real | Indisponível — Open-Meteo forecast cobre apenas **16 dias** (< 4 semanas de horizon) |
| Clima atual direto | Cria leakage: test e predição não teriam os dados na vida real |
| **Clima com lag 52w** | ✅ Sempre disponível: usa o clima do **mesmo período do ano passado** como proxy sazonal |

**Fluxo:**
```
Dados originais : Oct/2022 → Oct/2024
Fetch clima     : Oct/2021 → Oct/2023  (52 semanas antes)
Após shift +52w : Oct/2022 → Oct/2024  (alinhado com as séries)
Predição futura : Nov/2024 usa clima de Nov/2023  (histórico disponível ✅)
```

**Limitação:** não captura anomalias interanuais (El Niño, ondas de calor excepcionais).
Para pharma, o padrão sazonal é dominante — essa troca é aceitável.

**Output:** `data/gold/forecasting/features/ds_weather.parquet`  
**Chave:** `ds × region_id` — mesmo formato de `ds_holidays.parquet`

In [ ]:
import os
import time
import warnings
import requests
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../src')
from config import LoadConfig
cfg = LoadConfig()

In [ ]:
# Output
PATH_WEATHER = '../../data/gold/forecasting/features/ds_weather.parquet'
os.makedirs(os.path.dirname(PATH_WEATHER), exist_ok=True)

LAG_WEEKS    = 52    # shift temporal
HORIZON_WEEKS = 4   # semanas de previsão (para garantir cobertura futura)
DEMAND_TYPES = ['smooth', 'erratic', 'intermittent', 'lumpy']

# Coordenadas representativas por macro-região
# (capital ou cidade mais populosa da região)
REGION_COORDS = {
    1: {'name': 'Centro-Oeste', 'lat': -15.78, 'lon': -47.93},  # Brasília/DF
    2: {'name': 'Nordeste',     'lat':  -8.05, 'lon': -34.90},  # Recife/PE
    3: {'name': 'Norte',        'lat':  -3.10, 'lon': -60.02},  # Manaus/AM
    4: {'name': 'Sudeste',      'lat': -23.55, 'lon': -46.63},  # São Paulo/SP
    5: {'name': 'Sul',          'lat': -25.43, 'lon': -49.27},  # Curitiba/PR
}

DAILY_VARS = [
    'temperature_2m_max',
    'temperature_2m_min',
    'temperature_2m_mean',
    'precipitation_sum',
    'windspeed_10m_max',
]

## 1. Determina o intervalo de datas a buscar

In [ ]:
# Lê as semanas reais das séries refinadas (apenas unique_id + ds)
all_weeks = pd.concat([
    cfg.load_forecast('datasets', 'refined', dt)[['unique_id', 'ds']]
    for dt in DEMAND_TYPES
])['ds'].drop_duplicates().sort_values()

ds_min = all_weeks.min()   # início do dataset
ds_max = all_weeks.max()   # fim do dataset

# Intervalo a buscar na API = [ds_min - LAG_WEEKS, ds_max + HORIZON_WEEKS - LAG_WEEKS]
# Após shift +LAG_WEEKS, cobre [ds_min, ds_max + HORIZON_WEEKS]
fetch_start = ds_min  - pd.Timedelta(weeks=LAG_WEEKS)
fetch_end   = ds_max  + pd.Timedelta(weeks=HORIZON_WEEKS) - pd.Timedelta(weeks=LAG_WEEKS)

print(f'Dataset           : {ds_min.date()}  →  {ds_max.date()}')
print(f'Fetch clima (API) : {fetch_start.date()}  →  {fetch_end.date()}')
print(f'Após shift +{LAG_WEEKS}w   : {(fetch_start + pd.Timedelta(weeks=LAG_WEEKS)).date()}  →  {(fetch_end + pd.Timedelta(weeks=LAG_WEEKS)).date()}')
print(f'Semanas únicas    : {len(all_weeks):,}')
print(f'Dia da semana     : {all_weeks.dt.day_name().mode()[0]}  (ancoragem)')

## 2. Fetch Open-Meteo Historical Archive

In [ ]:
ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'


def fetch_daily_weather(lat: float, lon: float,
                         start: str, end: str,
                         variables: list,
                         retries: int = 3) -> pd.DataFrame:
    """
    Busca dados diários do Open-Meteo Historical Archive.
    Sem API key — serviço gratuito com rate limit suave.
    """
    params = {
        'latitude':  lat,
        'longitude': lon,
        'start_date': start,
        'end_date':   end,
        'daily':      ','.join(variables),
        'timezone':   'America/Sao_Paulo',
    }
    for attempt in range(retries):
        try:
            r = requests.get(ARCHIVE_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()['daily']
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['time'])
            return df.drop(columns='time').set_index('date')
        except requests.HTTPError as e:
            if r.status_code == 429:      # rate limit
                wait = 2 ** (attempt + 2)
                print(f'    Rate limit — aguardando {wait}s...')
                time.sleep(wait)
            else:
                raise
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise
    raise RuntimeError('Fetch falhou após todas as tentativas')


def aggregate_to_weekly(df_daily: pd.DataFrame, freq: str = 'W-MON') -> pd.DataFrame:
    """
    Agrega dados diários para semanal alinhado a W-MON (mesmo que o dataset).
    label='left' + closed='left': semana começa na segunda-feira.
    """
    weekly = df_daily.resample(freq, label='left', closed='left').agg({
        'temperature_2m_max':  'max',
        'temperature_2m_min':  'min',
        'temperature_2m_mean': 'mean',
        'precipitation_sum':   'sum',
        'windspeed_10m_max':   'max',
    }).rename(columns={
        'temperature_2m_max':  'temp_max',
        'temperature_2m_min':  'temp_min',
        'temperature_2m_mean': 'temp_mean',
        'precipitation_sum':   'precip',
        'windspeed_10m_max':   'wind_max',
    })
    # Features binárias de condição climática
    weekly['is_cold']  = (weekly['temp_mean'] < 15).astype(int)
    weekly['is_hot']   = (weekly['temp_mean'] > 28).astype(int)
    weekly['is_rainy'] = (weekly['precip']    > 10).astype(int)
    return weekly.reset_index().rename(columns={'date': 'ds'})

In [ ]:
start_str = fetch_start.strftime('%Y-%m-%d')
end_str   = fetch_end.strftime('%Y-%m-%d')

raw_by_region = {}

for region_id, info in REGION_COORDS.items():
    print(f'Buscando região {region_id} — {info["name"]} '
          f'({info["lat"]}, {info["lon"]})...', end=' ')
    df_daily  = fetch_daily_weather(info['lat'], info['lon'],
                                     start_str, end_str, DAILY_VARS)
    df_weekly = aggregate_to_weekly(df_daily)
    df_weekly['region_id'] = region_id
    raw_by_region[region_id] = df_weekly
    print(f'{len(df_weekly)} semanas OK')
    time.sleep(0.5)   # respeita rate limit suave

df_raw = pd.concat(raw_by_region.values(), ignore_index=True)
print(f'\nTotal: {len(df_raw):,} linhas  |  colunas: {df_raw.columns.tolist()}')

## 3. Aplica o shift de 52 semanas

```
weather_lag52[t] = clima real em (t - 52 semanas)
```

Equivalente a adiantar as datas do clima histórico em 52 semanas,
alinhando com as datas do modelo.

In [ ]:
WEATHER_COLS = ['temp_max', 'temp_min', 'temp_mean', 'precip', 'wind_max',
                'is_cold', 'is_hot', 'is_rainy']

df_weather = df_raw.copy()

# Avança datas em 52 semanas → alinha com o período real do modelo
df_weather['ds'] = df_weather['ds'] + pd.Timedelta(weeks=LAG_WEEKS)

# Filtra apenas semanas dentro do intervalo do dataset + horizon
valid_range_end = ds_max + pd.Timedelta(weeks=HORIZON_WEEKS)
df_weather = df_weather[
    (df_weather['ds'] >= ds_min) &
    (df_weather['ds'] <= valid_range_end)
].reset_index(drop=True)

# Ordena e verifica NaN
df_weather = df_weather[['ds', 'region_id'] + WEATHER_COLS].sort_values(['region_id', 'ds'])

print(f'Shape    : {df_weather.shape}')
print(f'Periodo  : {df_weather["ds"].min().date()}  →  {df_weather["ds"].max().date()}')
print(f'Regioes  : {sorted(df_weather["region_id"].unique())}')
print(f'\nNaN (%):')
print(df_weather[WEATHER_COLS].isna().mean().round(4))

## 4. Validação

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 7))

plot_vars = [
    ('temp_mean', 'Temp Media (C)',  'steelblue'),
    ('precip',    'Precipitacao (mm/semana)', 'royalblue'),
    ('wind_max',  'Vento Max (km/h)', 'slategray'),
    ('is_cold',   'Semana Fria (<15C)', 'lightblue'),
    ('is_hot',    'Semana Quente (>28C)', 'tomato'),
    ('is_rainy',  'Semana Chuvosa (>10mm)', 'cadetblue'),
]

for ax, (col, label, color) in zip(axes.flatten(), plot_vars):
    for region_id, info in REGION_COORDS.items():
        subset = df_weather[df_weather['region_id'] == region_id]
        ax.plot(subset['ds'], subset[col],
                label=info['name'], linewidth=1, alpha=0.8)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=8, frameon=False)
fig.suptitle(f'Clima histórico por macro-região (lag {LAG_WEEKS}w aplicado)',
             fontweight='bold', fontsize=11)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

# Sazonalidade mensal
print('\nTemp media mensal por região (°C) — padrão sazonal:')
df_weather['mes'] = df_weather['ds'].dt.month
pivot = df_weather.pivot_table(values='temp_mean', index='mes',
                                columns='region_id', aggfunc='mean').round(1)
pivot.columns = [REGION_COORDS[c]['name'] for c in pivot.columns]
print(pivot.to_string())
df_weather = df_weather.drop(columns='mes')

## 5. Salvar

In [ ]:
df_weather.to_parquet(PATH_WEATHER, index=False)

print(f'Salvo em : {PATH_WEATHER}')
print(f'Shape    : {df_weather.shape}')
print()
print('Colunas por grupo:')
print(f'  Chave       : ["ds", "region_id"]')
print(f'  Temperatura : ["temp_max", "temp_min", "temp_mean"]')
print(f'  Precipitação: ["precip"]')
print(f'  Vento       : ["wind_max"]')
print(f'  Binários    : ["is_cold", "is_hot", "is_rainy"]')
print()
print('Como usar no model_building:')
print('  df_weather = cfg.load_forecast("features", "weather")')
print('  df_train = df_train.merge(df_weather, on=["ds", "region_id"], how="left")')
print()
print('Para predicao futura — sem leakage:')
print(f'  Clima de Nov/2024 = clima real de Nov/2023  (já disponível no parquet)')